# Evaluate Qwen3 on HumanEval Benchmark

In [1]:
import os
import re
import signal
import traceback
import json
from contextlib import contextmanager
from io import StringIO
from typing import Optional

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/xz957/.conda/envs/sglang-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.0
NUM_SAMPLES = None  # None = all 164
TIMEOUT = 5
DTYPE = "auto"  # "auto", "float16", "bfloat16", "float32"
VERBOSE = True

## Sandboxed Execution

In [3]:
class TimeoutError(Exception):
    pass


@contextmanager
def time_limit(seconds: int):
    """Context manager to limit execution time (Unix only)."""
    def signal_handler(signum, frame):
        raise TimeoutError("Timed out!")
    signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)


def check_correctness(problem: dict, code: str, timeout: int = 5) -> dict:
    """Run the extracted code against the HumanEval test cases."""
    task_id = problem["task_id"]
    full_code = code + "\n\n" + problem["test"] + f"\ncheck({problem['entry_point']})\n"

    try:
        exec_globals = {}
        with time_limit(timeout):
            exec(full_code, exec_globals)
        return {"task_id": task_id, "passed": True, "error": None}
    except TimeoutError:
        return {"task_id": task_id, "passed": False, "error": "Timed out"}
    except Exception as e:
        return {"task_id": task_id, "passed": False, "error": str(e)}

## Completion Extraction & Prompt Formatting

In [4]:
def extract_code(generated_text: str) -> str:
    """Extract python code from model output. Looks for ```python blocks, falls back to raw text."""
    match = re.search(r"```(?:python)?\s*\n(.*?)```", generated_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return generated_text.strip()


def format_prompt(problem_prompt: str) -> str:
    """Wrap the HumanEval prompt with a chat instruction."""
    instruction = (
        "Complete the following Python function. "
        "Return ONLY the complete function implementation in a python code block.\n\n"
    )
    return instruction + problem_prompt

## Load Model & Dataset

In [5]:
# Device setup
if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

dtype_map = {"auto": "auto", "float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
torch_dtype = dtype_map[DTYPE]

print(f"Model: {MODEL_NAME} | Device: {device} | Dtype: {DTYPE}")
print(f"Max tokens: {MAX_NEW_TOKENS} | Temperature: {TEMPERATURE} | Timeout: {TIMEOUT}s")

Model: Qwen/Qwen3-8B | Device: cuda | Dtype: auto
Max tokens: 2048 | Temperature: 0.0 | Timeout: 5s


In [6]:
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch_dtype,
    device_map=device if device == "cuda" else None,
    trust_remote_code=True,
)
if device != "cuda":
    model = model.to(device)
model.eval()
print(f"Model loaded on {device}")

Loading model and tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|█| 5/5 [00:35<00:00,  7.04s/it

Model loaded on cuda


In [7]:
print("Loading HumanEval dataset...")
dataset = load_dataset("openai_humaneval", split="test")
if NUM_SAMPLES is not None:
    dataset = dataset.select(range(min(NUM_SAMPLES, len(dataset))))
print(f"{len(dataset)} problems loaded")

Loading HumanEval dataset...
164 problems loaded


## Run Evaluation

In [8]:
# Quick sanity check: run one problem and inspect raw output
problem = dataset[0]
formatted_prompt = format_prompt(problem["prompt"])

messages = [{"role": "user", "content": formatted_prompt}]
input_text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs = tokenizer(input_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
code = extract_code(generated_text)

print(f"Task: {problem['task_id']}")
print(f"--- Raw Model Output ---")
print(generated_text[:2000])
print(f"\n--- Extracted Code ---")
print(code)
print(f"\n--- Test Result ---")
print(check_correctness(problem, code, timeout=TIMEOUT))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Task: HumanEval/0
--- Raw Model Output ---
```python
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False
```

--- Extracted Code ---
from typing import List

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    for i in range(len(numbers)):
        fo

In [9]:
results = []
passed = 0

for i, problem in enumerate(dataset):
    task_id = problem["task_id"]
    formatted_prompt = format_prompt(problem["prompt"])

    messages = [{"role": "user", "content": formatted_prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    gen_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": TEMPERATURE > 0,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if TEMPERATURE > 0:
        gen_kwargs["temperature"] = TEMPERATURE

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    code = extract_code(generated_text)

    result = check_correctness(problem, code, timeout=TIMEOUT)
    results.append(result)

    if result["passed"]:
        passed += 1

    status = "PASS" if result["passed"] else "FAIL"
    acc_so_far = passed / (i + 1) * 100

    if VERBOSE:
        print(f"[{i+1}/{len(dataset)}] {task_id}: {status}")
        if result["error"]:
            print(f"  Error: {result['error']}")
        # print(f"  Code: {code[:80].strip()}...")
    else:
        print(f"[{i+1}/{len(dataset)}] {task_id}: {status}  (running pass@1: {acc_so_far:.1f}%)")

[1/164] HumanEval/0: PASS
[2/164] HumanEval/1: PASS
[3/164] HumanEval/2: PASS
[4/164] HumanEval/3: PASS
[5/164] HumanEval/4: PASS
[6/164] HumanEval/5: PASS
[7/164] HumanEval/6: PASS
[8/164] HumanEval/7: PASS
[9/164] HumanEval/8: PASS
[10/164] HumanEval/9: PASS
[11/164] HumanEval/10: FAIL
  Error: name 'is_palindrome' is not defined
[12/164] HumanEval/11: PASS
[13/164] HumanEval/12: PASS
[14/164] HumanEval/13: PASS
[15/164] HumanEval/14: PASS
[16/164] HumanEval/15: PASS
[17/164] HumanEval/16: PASS
[18/164] HumanEval/17: PASS
[19/164] HumanEval/18: PASS
[20/164] HumanEval/19: PASS
[21/164] HumanEval/20: PASS
[22/164] HumanEval/21: PASS
[23/164] HumanEval/22: PASS
[24/164] HumanEval/23: PASS
[25/164] HumanEval/24: PASS
[26/164] HumanEval/25: PASS
[27/164] HumanEval/26: FAIL
[28/164] HumanEval/27: PASS
[29/164] HumanEval/28: PASS
[30/164] HumanEval/29: PASS
[31/164] HumanEval/30: PASS
[32/164] HumanEval/31: PASS
[33/164] HumanEval/32: FAIL
[34/164] HumanEval/33: PASS
[35/164] HumanEval/34:

## Results Summary

In [11]:
total = len(results)
accuracy = passed / total * 100 if total > 0 else 0.0

print(f"Model: {MODEL_NAME}")
print(f"Total: {total} | Passed: {passed} | Failed: {total - passed}")
print(f"Pass@1: {accuracy:.2f}%")

errors = [r for r in results if not r["passed"]]
if errors:
    timeout_count = sum(1 for r in errors if r["error"] == "Timed out")
    runtime_count = len(errors) - timeout_count
    print(f"\nError breakdown: Timeouts={timeout_count}, Runtime errors={runtime_count}")

Model: Qwen/Qwen3-8B
Total: 164 | Passed: 138 | Failed: 26
Pass@1: 84.15%

Error breakdown: Timeouts=0, Runtime errors=26
